# Fase 5 — Streamlit final conectado a FastAPI + RAG v2.2

Esta versión final de Streamlit consume los servicios de FastAPI ya
productivizados en la Fase 5.1:

- `/predict` → XGBoost + SHAP.
- `/rag` → retrieval v2.2 + prompt v3 + `mistral:7b`.

La aplicación **no reconstruye ni redefine el RAG**. El backend es la única
fuente de verdad para la lógica documental.

## 1. Configuración y recursos del proyecto


Antes de generar los módulos se comprueba la presencia de los recursos persistentes creados en fases anteriores.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()

SRC_DIR = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"
VECTORSTORE_DIR = DATA_DIR / "vectorstore" / "chroma_vut_malaga_v2"

MODEL_PATH = MODELS_DIR / "xgboost_model_final.pkl"
FEATURES_PATH = PROCESSED_DIR / "listings_features.parquet"
GEOJSON_PATH = RAW_DIR / "neighbourhoods.geojson"

SRC_DIR.mkdir(parents=True, exist_ok=True)

required_resources = {
    "Modelo final": MODEL_PATH,
    "Features": FEATURES_PATH,
    "GeoJSON": GEOJSON_PATH,
    "Vectorstore Chroma": VECTORSTORE_DIR,
}

for name, path in required_resources.items():
    symbol = "✅" if path.exists() else "❌"
    print(f"{symbol} {name}: {path}")

missing = [
    name
    for name, path in required_resources.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Faltan recursos necesarios: "
        + ", ".join(missing)
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 2. Arquitectura modular

Para separar la interfaz de la lógica interna, la aplicación se organiza en
varios módulos reutilizables:

- `geo_utils.py`: variables geoespaciales;
- `model_utils.py`: preparación de features, predicción, SHAP y contexto cartográfico;
- `rag_utils.py`: retrieval híbrido, contexto documental y generación con Ollama.

`app.py` consume estos módulos y únicamente se encarga de la interfaz.

## 3. Módulo geoespacial

Se reutiliza la lógica de la Fase 2 para garantizar que las variables calculadas para una nueva vivienda sean compatibles con las utilizadas durante el entrenamiento.

In [ ]:
GEO_UTILS_PATH = SRC_DIR / "geo_utils.py"

GEO_UTILS_CODE = '\nimport re\nimport unicodedata\n\nimport h3\nimport numpy as np\nimport pandas as pd\nimport geopandas as gpd\n\nfrom pyproj import Geod\nfrom shapely.geometry import Point\n\n\n# ============================================================\n# POIs UTILIZADOS EN EL ENTRENAMIENTO\n# ============================================================\n\nPOIS = {\n    "malagueta": (\n        36.7196,\n        -4.4087,\n        "beach",\n    ),\n    "pedregalejo": (\n        36.7209,\n        -4.3699,\n        "beach",\n    ),\n    "el_palo": (\n        36.7169,\n        -4.3565,\n        "beach",\n    ),\n    "calle_larios": (\n        36.7196,\n        -4.4211,\n        "historic_center",\n    ),\n    "plaza_constitucion": (\n        36.7202,\n        -4.4218,\n        "historic_center",\n    ),\n    "museo_picasso": (\n        36.7217,\n        -4.4185,\n        "culture",\n    ),\n    "alcazaba": (\n        36.7213,\n        -4.4165,\n        "culture",\n    ),\n    "centro_pompidou": (\n        36.7182,\n        -4.4130,\n        "culture",\n    ),\n    "maria_zambrano": (\n        36.7111,\n        -4.4314,\n        "transport",\n    ),\n    "aeropuerto_agp": (\n        36.6749,\n        -4.4991,\n        "transport",\n    ),\n}\n\n\nGEOD = Geod(\n    ellps="WGS84"\n)\n\n\n# ============================================================\n# UTILIDADES\n# ============================================================\n\ndef normalize_column_name(column_name):\n\n    text = str(column_name).strip().lower()\n\n    text = unicodedata.normalize(\n        "NFKD",\n        text,\n    )\n\n    text = "".join(\n        character\n        for character in text\n        if not unicodedata.combining(character)\n    )\n\n    return re.sub(\n        r"[^a-z0-9]+",\n        "_",\n        text,\n    ).strip("_")\n\n\ndef mode_or_nan(series):\n\n    series = series.dropna()\n\n    if len(series) == 0:\n        return np.nan\n\n    return series.mode().iloc[0]\n\n\n# ============================================================\n# PREPARACIÓN DE RECURSOS\n# ============================================================\n\ndef prepare_geospatial_resources(\n    districts_file,\n    df_features,\n):\n\n    gdf_raw = gpd.read_file(\n        districts_file\n    )\n\n    normalized_column_map = {\n        normalize_column_name(column): column\n        for column in gdf_raw.columns\n    }\n\n    district_name_candidates = [\n        "neighbourhood",\n        "neighbourhood_group",\n        "nombre",\n        "distrito",\n        "nom_distrito",\n        "nombre_distrito",\n        "descrip",\n        "descripcion",\n        "name",\n    ]\n\n    district_name_column = None\n\n    for candidate in district_name_candidates:\n\n        if candidate in normalized_column_map:\n\n            district_name_column = (\n                normalized_column_map[candidate]\n            )\n\n            break\n\n    if district_name_column is None:\n        raise KeyError(\n            "No se ha podido detectar la "\n            "columna de distrito."\n        )\n\n    gdf_districts = (\n        gdf_raw[\n            [\n                district_name_column,\n                gdf_raw.geometry.name,\n            ]\n        ]\n        .rename(\n            columns={\n                district_name_column:\n                "district"\n            }\n        )\n        .copy()\n    )\n\n    gdf_districts["district"] = (\n        gdf_districts["district"]\n        .astype(str)\n        .str.strip()\n    )\n\n    if gdf_districts.crs is None:\n\n        gdf_districts = (\n            gdf_districts\n            .set_crs("EPSG:4326")\n        )\n\n    gdf_districts = (\n        gdf_districts\n        .to_crs("EPSG:4326")\n    )\n\n    # Elimina dimensión Z si existe\n    try:\n        import shapely\n\n        gdf_districts["geometry"] = (\n            gdf_districts.geometry.apply(\n                lambda geometry:\n                shapely.force_2d(geometry)\n                if getattr(\n                    geometry,\n                    "has_z",\n                    False,\n                )\n                else geometry\n            )\n        )\n    except Exception:\n        pass\n\n    # --------------------------------------------------------\n    # ÁREA DE DISTRITOS\n    # --------------------------------------------------------\n\n    metric = (\n        gdf_districts\n        .to_crs("EPSG:25830")\n        .copy()\n    )\n\n    metric[\n        "district_area_km2"\n    ] = (\n        metric.geometry.area\n        / 1_000_000\n    )\n\n    district_area_map = (\n        metric\n        .set_index("district")[\n            "district_area_km2"\n        ]\n        .to_dict()\n    )\n\n    # --------------------------------------------------------\n    # FRECUENCIAS DEL SNAPSHOT\n    # --------------------------------------------------------\n\n    district_count_map = (\n        df_features\n        .groupby("district")\n        .size()\n        .to_dict()\n    )\n\n    neighbourhood_map = (\n        df_features\n        .groupby("district")[\n            "neighbourhood_cleansed"\n        ]\n        .agg(mode_or_nan)\n        .to_dict()\n    )\n\n    neighbourhood_group_map = (\n        df_features\n        .groupby("district")[\n            "neighbourhood_group"\n        ]\n        .agg(mode_or_nan)\n        .to_dict()\n    )\n\n    h3_count_maps = {}\n\n    for resolution in [8, 9]:\n\n        column = (\n            f"h3_res{resolution}"\n        )\n\n        h3_count_maps[\n            resolution\n        ] = (\n            df_features[column]\n            .value_counts()\n            .to_dict()\n        )\n\n    return {\n        "gdf_districts": gdf_districts,\n        "district_area_map":\n            district_area_map,\n        "district_count_map":\n            district_count_map,\n        "neighbourhood_map":\n            neighbourhood_map,\n        "neighbourhood_group_map":\n            neighbourhood_group_map,\n        "h3_count_maps":\n            h3_count_maps,\n    }\n\n\n# ============================================================\n# CONSTRUCCIÓN DE FEATURES GEOESPACIALES\n# ============================================================\n\ndef build_geospatial_features(\n    latitude,\n    longitude,\n    resources,\n    new_listing=True,\n):\n\n    features = {\n        "latitude": float(latitude),\n        "longitude": float(longitude),\n    }\n\n    gdf_districts = (\n        resources["gdf_districts"]\n    )\n\n    # --------------------------------------------------------\n    # DISTRITO\n    # --------------------------------------------------------\n\n    point_gdf = gpd.GeoDataFrame(\n        {\n            "latitude": [latitude],\n            "longitude": [longitude],\n        },\n        geometry=[\n            Point(\n                float(longitude),\n                float(latitude),\n            )\n        ],\n        crs="EPSG:4326",\n    )\n\n    joined = gpd.sjoin(\n        point_gdf,\n        gdf_districts[\n            [\n                "district",\n                "geometry",\n            ]\n        ],\n        how="left",\n        predicate="within",\n    )\n\n    district = (\n        joined.iloc[0]["district"]\n    )\n\n    if pd.isna(district):\n\n        raise ValueError(\n            "La ubicación no pertenece a "\n            "ningún distrito disponible."\n        )\n\n    features["district"] = district\n\n    features[\n        "neighbourhood_cleansed"\n    ] = (\n        resources[\n            "neighbourhood_map"\n        ].get(\n            district,\n            district,\n        )\n    )\n\n    features[\n        "neighbourhood_group"\n    ] = (\n        resources[\n            "neighbourhood_group_map"\n        ].get(\n            district,\n            district,\n        )\n    )\n\n    # --------------------------------------------------------\n    # DENSIDAD DISTRITO\n    # --------------------------------------------------------\n\n    existing_count = int(\n        resources[\n            "district_count_map"\n        ].get(\n            district,\n            0,\n        )\n    )\n\n    listing_count = (\n        existing_count + 1\n        if new_listing\n        else existing_count\n    )\n\n    district_area = float(\n        resources[\n            "district_area_map"\n        ][district]\n    )\n\n    features[\n        "district_listing_count"\n    ] = listing_count\n\n    features[\n        "district_area_km2"\n    ] = district_area\n\n    features[\n        "district_listing_density_km2"\n    ] = (\n        listing_count\n        / district_area\n    )\n\n    # --------------------------------------------------------\n    # H3\n    # --------------------------------------------------------\n\n    for resolution in [8, 9]:\n\n        cell = h3.latlng_to_cell(\n            float(latitude),\n            float(longitude),\n            resolution,\n        )\n\n        features[\n            f"h3_res{resolution}"\n        ] = cell\n\n        existing_h3_count = int(\n            resources[\n                "h3_count_maps"\n            ][resolution].get(\n                cell,\n                0,\n            )\n        )\n\n        if new_listing:\n\n            h3_listing_count = (\n                existing_h3_count + 1\n            )\n\n            competitor_count = (\n                existing_h3_count\n            )\n\n        else:\n\n            h3_listing_count = (\n                existing_h3_count\n            )\n\n            competitor_count = max(\n                h3_listing_count - 1,\n                0,\n            )\n\n        area = h3.cell_area(\n            cell,\n            unit="km^2",\n        )\n\n        features[\n            f"h3_res{resolution}_listing_count"\n        ] = h3_listing_count\n\n        features[\n            f"h3_res{resolution}_competitor_count"\n        ] = competitor_count\n\n        features[\n            f"h3_res{resolution}_area_km2"\n        ] = area\n\n        features[\n            f"h3_res{resolution}_density_km2"\n        ] = (\n            h3_listing_count\n            / area\n        )\n\n    # --------------------------------------------------------\n    # DISTANCIAS POI\n    # --------------------------------------------------------\n\n    for poi_name, (\n        poi_latitude,\n        poi_longitude,\n        poi_category,\n    ) in POIS.items():\n\n        _, _, distance_m = GEOD.inv(\n            float(longitude),\n            float(latitude),\n            float(poi_longitude),\n            float(poi_latitude),\n        )\n\n        features[\n            f"distance_{poi_name}_km"\n        ] = (\n            distance_m / 1000\n        )\n\n    # --------------------------------------------------------\n    # POI MÁS CERCANO\n    # --------------------------------------------------------\n\n    categories = sorted({\n        values[2]\n        for values in POIS.values()\n    })\n\n    for category in categories:\n\n        category_pois = [\n            name\n            for name, values\n            in POIS.items()\n            if values[2] == category\n        ]\n\n        distances = {\n            name: features[\n                f"distance_{name}_km"\n            ]\n            for name in category_pois\n        }\n\n        nearest_poi = min(\n            distances,\n            key=distances.get,\n        )\n\n        features[\n            f"distance_nearest_{category}_km"\n        ] = distances[\n            nearest_poi\n        ]\n\n        features[\n            f"nearest_{category}_poi"\n        ] = nearest_poi\n\n    # --------------------------------------------------------\n    # INDICADORES DE PROXIMIDAD\n    # --------------------------------------------------------\n\n    distance_thresholds = {\n        "within_500m_beach": (\n            "distance_nearest_beach_km",\n            0.5,\n        ),\n        "within_1km_beach": (\n            "distance_nearest_beach_km",\n            1.0,\n        ),\n        "within_1km_historic_center": (\n            "distance_nearest_historic_center_km",\n            1.0,\n        ),\n        "within_2km_historic_center": (\n            "distance_nearest_historic_center_km",\n            2.0,\n        ),\n        "within_1km_culture": (\n            "distance_nearest_culture_km",\n            1.0,\n        ),\n        "within_1km_transport": (\n            "distance_nearest_transport_km",\n            1.0,\n        ),\n        "within_5km_airport": (\n            "distance_aeropuerto_agp_km",\n            5.0,\n        ),\n    }\n\n    for new_column, (\n        distance_column,\n        threshold,\n    ) in distance_thresholds.items():\n\n        features[new_column] = int(\n            features[\n                distance_column\n            ]\n            <= threshold\n        )\n\n    # --------------------------------------------------------\n    # LOG1P\n    # --------------------------------------------------------\n\n    distance_columns = [\n        column\n        for column in list(\n            features.keys()\n        )\n        if (\n            column.startswith(\n                "distance_"\n            )\n            and column.endswith(\n                "_km"\n            )\n        )\n    ]\n\n    for column in distance_columns:\n\n        features[\n            f"log1p_{column}"\n        ] = np.log1p(\n            features[column]\n        )\n\n    return features\n'

GEO_UTILS_PATH.write_text(
    GEO_UTILS_CODE,
    encoding="utf-8",
)

print(GEO_UTILS_PATH)

## 4. Módulo predictivo

El módulo contiene la clase necesaria para deserializar el pipeline final, construye exactamente las variables esperadas por el modelo, realiza la predicción directamente en euros y genera explicaciones SHAP.

El modelo de producción es **XGBoost**, entrenado directamente sobre `price` con `reg:absoluteerror`. Por tanto, no se aplica ninguna transformación `log1p`/`expm1` durante la inferencia.

También incorpora el contexto necesario para el mapa: distrito, celda H3, competidores de la misma celda y alojamientos incluidos en un radio configurable.


In [ ]:
MODEL_UTILS_PATH = SRC_DIR / "model_utils.py"

MODEL_UTILS_CODE = '\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom pyproj import Geod\nimport shap\n\nfrom sklearn.base import BaseEstimator, TransformerMixin\n\nfrom src.geo_utils import (\n    prepare_geospatial_resources,\n    build_geospatial_features,\n)\n\nGEOD = Geod(ellps="WGS84")\n\n\n# ============================================================\n# TRANSFORMER NECESARIO PARA CARGAR EL PIPELINE\n# ============================================================\n\nclass H3TargetMeanEncoder(BaseEstimator, TransformerMixin):\n\n    def __init__(\n        self,\n        h3_columns=("h3_res8", "h3_res9"),\n        smoothing=11.720095,\n        drop_original=True,\n    ):\n        self.h3_columns = h3_columns\n        self.smoothing = smoothing\n        self.drop_original = drop_original\n\n    def fit(self, X, y):\n\n        X = (\n            pd.DataFrame(X)\n            .reset_index(drop=True)\n            .copy()\n        )\n\n        y = (\n            pd.Series(y)\n            .reset_index(drop=True)\n        )\n\n        self.global_mean_ = float(\n            y.mean()\n        )\n\n        self.encoding_maps_ = {}\n\n        for col in self.h3_columns:\n\n            if col not in X.columns:\n                continue\n\n            tmp = pd.DataFrame({\n                "group": (\n                    X[col]\n                    .astype("string")\n                    .fillna("H3_desconocido")\n                ),\n                "target": y,\n            })\n\n            stats = (\n                tmp.groupby("group")["target"]\n                .agg(["mean", "count"])\n            )\n\n            smooth = (\n                stats["count"] * stats["mean"]\n                + self.smoothing\n                * self.global_mean_\n            ) / (\n                stats["count"]\n                + self.smoothing\n            )\n\n            self.encoding_maps_[col] = (\n                smooth.to_dict()\n            )\n\n        return self\n\n    def transform(self, X):\n\n        X = pd.DataFrame(X).copy()\n\n        for col in self.h3_columns:\n\n            if col not in X.columns:\n                continue\n\n            X[\n                f"{col}_mean_price_te"\n            ] = (\n                X[col]\n                .astype("string")\n                .fillna("H3_desconocido")\n                .map(\n                    self.encoding_maps_\n                    .get(col, {})\n                )\n                .fillna(\n                    self.global_mean_\n                )\n                .astype(float)\n            )\n\n        if self.drop_original:\n\n            X = X.drop(\n                columns=[\n                    col\n                    for col\n                    in self.h3_columns\n                    if col in X.columns\n                ],\n                errors="ignore",\n            )\n\n        return X\n\n\n# ============================================================\n# RUTAS DEL PROYECTO\n# ============================================================\n\nPROJECT_ROOT = (\n    Path(__file__)\n    .resolve()\n    .parent\n    .parent\n)\n\nMODEL_PATH = (\n    PROJECT_ROOT\n    / "models"\n    / "xgboost_model_final.pkl"\n)\n\nFEATURES_PATH = (\n    PROJECT_ROOT\n    / "data"\n    / "processed"\n    / "listings_features.parquet"\n)\n\nDISTRICTS_FILE = (\n    PROJECT_ROOT\n    / "data"\n    / "raw"\n    / "neighbourhoods.geojson"\n)\n\n\n# ============================================================\n# CARGA DE DATOS Y MODELO\n# ============================================================\n\ndf_features = pd.read_parquet(\n    FEATURES_PATH\n)\n\nX_app = (\n    df_features\n    .drop(\n        columns=["price"],\n        errors="ignore",\n    )\n    .copy()\n)\n\n\n# Importante:\n# el pickle original referencia H3TargetMeanEncoder\n# como clase definida en __main__.\n#\n# Para mantener compatibilidad con el modelo ya guardado,\n# exponemos temporalmente la clase en __main__ antes de cargar.\n\nimport __main__\n\n__main__.H3TargetMeanEncoder = (\n    H3TargetMeanEncoder\n)\n\n\nmodel = joblib.load(\n    MODEL_PATH\n)\n\n\n# ============================================================\n# RECURSOS GEOESPACIALES\n# ============================================================\n\ngeo_resources = (\n    prepare_geospatial_resources(\n        districts_file=DISTRICTS_FILE,\n        df_features=df_features,\n    )\n)\n\n\n# ============================================================\n# VARIABLES DE LA INTERFAZ\n# ============================================================\n\nUSER_AMENITIES = [\n    "has_wifi",\n    "has_kitchen",\n    "has_air_conditioning",\n    "has_heating",\n    "has_parking",\n    "has_pool",\n    "has_washer",\n    "has_dryer",\n    "has_tv",\n    "has_balcony_or_terrace",\n    "has_sea_view",\n    "has_workspace",\n    "has_elevator",\n    "has_pets_allowed",\n    "has_crib",\n    "has_bbq",\n    "has_gym",\n    "has_hot_tub",\n    "has_breakfast",\n]\n\n\nHISTORICAL_COLUMNS = [\n    "host_listings_count",\n    "number_of_reviews",\n    "number_of_reviews_ltm",\n    "review_scores_rating",\n    "review_scores_cleanliness",\n    "review_scores_location",\n    "availability_30",\n    "availability_365",\n]\n\n\nhistorical_defaults = {\n    col: X_app[col].median()\n    for col in HISTORICAL_COLUMNS\n}\n\n\n# ============================================================\n# REFERENCIAS INTERNAS DEL PIPELINE\n# ============================================================\n\nh3_encoder = (\n    model.named_steps[\n        "h3_target_encoder"\n    ]\n)\n\npreprocessor = (\n    model.named_steps[\n        "preprocessor"\n    ]\n)\n\nvariance_filter = (\n    model.named_steps[\n        "variance_filter"\n    ]\n)\n\nxgboost_model = (\n    model.named_steps[\n        "model"\n    ]\n)\n\n\nfeature_names_pre = (\n    preprocessor\n    .get_feature_names_out()\n)\n\nvariance_mask = (\n    variance_filter\n    .get_support()\n)\n\nfeature_names_final = (\n    feature_names_pre[\n        variance_mask\n    ]\n)\n\n\nexplainer = shap.TreeExplainer(\n    xgboost_model\n)\n\n\n# ============================================================\n# CONSTRUCCIÓN DE LAS 164 VARIABLES\n# ============================================================\n\ndef build_listing_features(\n    latitude,\n    longitude,\n    room_type,\n    property_type_group,\n    accommodates,\n    bedrooms,\n    bathrooms_num,\n    minimum_nights,\n    maximum_nights,\n    instant_bookable,\n    host_is_superhost,\n    selected_amenities=None,\n):\n\n    if selected_amenities is None:\n        selected_amenities = []\n\n    new_row = {}\n\n    # --------------------------------------------------------\n    # Valores neutrales iniciales\n    # --------------------------------------------------------\n\n    numeric_columns = (\n        X_app\n        .select_dtypes(\n            include=["number"]\n        )\n        .columns\n    )\n\n    for col in numeric_columns:\n        new_row[col] = (\n            X_app[col].median()\n        )\n\n    categorical_columns = (\n        X_app\n        .select_dtypes(\n            include=[\n                "object",\n                "string",\n                "category",\n                "bool",\n            ]\n        )\n        .columns\n    )\n\n    for col in categorical_columns:\n\n        mode = (\n            X_app[col]\n            .mode(dropna=True)\n        )\n\n        new_row[col] = (\n            mode.iloc[0]\n            if len(mode) > 0\n            else np.nan\n        )\n\n    # --------------------------------------------------------\n    # Datos del usuario\n    # --------------------------------------------------------\n\n    new_row.update({\n        "latitude":\n            float(latitude),\n\n        "longitude":\n            float(longitude),\n\n        "room_type":\n            room_type,\n\n        "property_type_group":\n            property_type_group,\n\n        "accommodates":\n            int(accommodates),\n\n        "bedrooms":\n            float(bedrooms),\n\n        "bathrooms_num":\n            float(bathrooms_num),\n\n        "minimum_nights":\n            int(minimum_nights),\n\n        "maximum_nights":\n            int(maximum_nights),\n\n        "instant_bookable":\n            int(instant_bookable),\n\n        "host_is_superhost":\n            int(host_is_superhost),\n    })\n\n    # --------------------------------------------------------\n    # Variables históricas\n    # --------------------------------------------------------\n\n    for col, value in (\n        historical_defaults.items()\n    ):\n        new_row[col] = value\n\n    # --------------------------------------------------------\n    # Variables geoespaciales\n    # --------------------------------------------------------\n\n    geo_features = (\n        build_geospatial_features(\n            latitude=latitude,\n            longitude=longitude,\n            resources=geo_resources,\n            new_listing=True,\n        )\n    )\n\n    new_row.update(\n        geo_features\n    )\n\n    # --------------------------------------------------------\n    # Amenities manuales\n    # --------------------------------------------------------\n\n    for col in USER_AMENITIES:\n\n        new_row[col] = int(\n            col in selected_amenities\n        )\n\n    new_row[\n        "amenities_count"\n    ] = len(\n        selected_amenities\n    )\n\n    # --------------------------------------------------------\n    # Amenities automáticas\n    # --------------------------------------------------------\n\n    amenity_auto_columns = [\n        col\n        for col in X_app.columns\n        if col.startswith(\n            "amenity_auto_"\n        )\n    ]\n\n    for col in amenity_auto_columns:\n        new_row[col] = 0\n\n\n    amenity_auto_map = {\n\n        "has_wifi":\n            "amenity_auto_wifi",\n\n        "has_kitchen":\n            "amenity_auto_kitchen",\n\n        "has_air_conditioning":\n            "amenity_auto_air_conditioning",\n\n        "has_heating":\n            "amenity_auto_heating",\n\n        "has_washer":\n            "amenity_auto_washer",\n\n        "has_tv":\n            "amenity_auto_tv",\n\n        "has_elevator":\n            "amenity_auto_elevator",\n\n        "has_workspace":\n            "amenity_auto_dedicated_workspace",\n\n        "has_crib":\n            "amenity_auto_crib",\n    }\n\n\n    for (\n        manual_col,\n        auto_col,\n    ) in amenity_auto_map.items():\n\n        if (\n            manual_col\n            in selected_amenities\n            and auto_col\n            in new_row\n        ):\n            new_row[\n                auto_col\n            ] = 1\n\n\n    new_row[\n        "amenities_count_phase2"\n    ] = sum(\n        new_row[col]\n        for col\n        in amenity_auto_columns\n    )\n\n\n    # --------------------------------------------------------\n    # Variables derivadas\n    # --------------------------------------------------------\n\n    new_row[\n        "has_reviews"\n    ] = int(\n        new_row[\n            "number_of_reviews"\n        ] > 0\n    )\n\n\n    # --------------------------------------------------------\n    # Orden exacto esperado por el modelo\n    # --------------------------------------------------------\n\n    X_new = pd.DataFrame(\n        [new_row]\n    )\n\n    X_new = (\n        X_new\n        .reindex(\n            columns=X_app.columns\n        )\n    )\n\n    return X_new\n\n\n# ============================================================\n# PREDICCIÓN FINAL\n# ============================================================\n\ndef predict_listing(\n    latitude,\n    longitude,\n    room_type,\n    property_type_group,\n    accommodates,\n    bedrooms,\n    bathrooms_num,\n    minimum_nights,\n    maximum_nights,\n    instant_bookable,\n    host_is_superhost,\n    selected_amenities=None,\n):\n\n    X_new = build_listing_features(\n        latitude=latitude,\n        longitude=longitude,\n        room_type=room_type,\n        property_type_group=property_type_group,\n        accommodates=accommodates,\n        bedrooms=bedrooms,\n        bathrooms_num=bathrooms_num,\n        minimum_nights=minimum_nights,\n        maximum_nights=maximum_nights,\n        instant_bookable=instant_bookable,\n        host_is_superhost=host_is_superhost,\n        selected_amenities=selected_amenities,\n    )\n\n    # --------------------------------------------------------\n    # Predicción\n    # --------------------------------------------------------\n\n    pred_price = float(\n        model.predict(\n            X_new\n        )[0]\n    )\n\n    # --------------------------------------------------------\n    # Transformación para SHAP\n    # --------------------------------------------------------\n\n    X_h3 = (\n        h3_encoder\n        .transform(\n            X_new\n        )\n    )\n\n    X_pre = (\n        preprocessor\n        .transform(\n            X_h3\n        )\n    )\n\n    X_var = (\n        variance_filter\n        .transform(\n            X_pre\n        )\n    )\n\n    X_var_dense = (\n        X_var.toarray()\n        if hasattr(\n            X_var,\n            "toarray",\n        )\n        else np.asarray(\n            X_var\n        )\n    )\n\n    # --------------------------------------------------------\n    # SHAP\n    # --------------------------------------------------------\n\n    shap_values = (\n        explainer(\n            X_var_dense\n        )\n    )\n\n    shap_explanation = (\n        shap.Explanation(\n            values=(\n                shap_values\n                .values[0]\n            ),\n            base_values=(\n                shap_values\n                .base_values[0]\n            ),\n            data=(\n                X_var_dense[0]\n            ),\n            feature_names=(\n                feature_names_final\n            ),\n        )\n    )\n\n    return {\n        "price":\n            pred_price,\n\n        "features":\n            X_new,\n\n        "shap_explanation":\n            shap_explanation,\n    }\n\n# ============================================================\n# DATOS PARA EL VISOR CARTOGRÁFICO\n# ============================================================\n\ndef get_map_context(\n    latitude,\n    longitude,\n    radius_km=1.0,\n):\n    """\n    Devuelve la información necesaria para representar la vivienda,\n    su distrito, su celda H3 res. 8, los competidores de dicha celda\n    y los alojamientos situados dentro de un radio determinado.\n    """\n\n    latitude = float(latitude)\n    longitude = float(longitude)\n    radius_km = float(radius_km)\n\n    geo_features = build_geospatial_features(\n        latitude=latitude,\n        longitude=longitude,\n        resources=geo_resources,\n        new_listing=True,\n    )\n\n    district = geo_features["district"]\n    h3_res8 = geo_features["h3_res8"]\n\n    district_gdf = (\n        geo_resources["gdf_districts"]\n        .loc[\n            geo_resources["gdf_districts"]["district"] == district\n        ]\n        .copy()\n    )\n\n    competitors = (\n        df_features\n        .loc[\n            df_features["h3_res8"] == h3_res8,\n            [\n                "latitude",\n                "longitude",\n                "price",\n                "room_type",\n                "property_type_group",\n            ],\n        ]\n        .copy()\n    )\n\n    nearby = df_features[\n        [\n            "latitude",\n            "longitude",\n            "price",\n            "room_type",\n            "property_type_group",\n        ]\n    ].copy()\n\n    _, _, distances_m = GEOD.inv(\n        np.full(len(nearby), longitude),\n        np.full(len(nearby), latitude),\n        nearby["longitude"].to_numpy(),\n        nearby["latitude"].to_numpy(),\n    )\n\n    nearby["distance_km"] = distances_m / 1000.0\n\n    nearby = (\n        nearby\n        .loc[nearby["distance_km"] <= radius_km]\n        .sort_values("distance_km")\n        .copy()\n    )\n\n    return {\n        "latitude": latitude,\n        "longitude": longitude,\n        "district": district,\n        "h3_res8": h3_res8,\n        "district_gdf": district_gdf,\n        "competitors": competitors,\n        "nearby": nearby,\n        "radius_km": radius_km,\n    }\n'

MODEL_UTILS_PATH.write_text(
    MODEL_UTILS_CODE,
    encoding="utf-8",
)

print(MODEL_UTILS_PATH)

## 5. Validación del motor predictivo

Se realiza una única prueba integrada.

In [ ]:
import importlib

import src.geo_utils
import src.model_utils

importlib.reload(src.geo_utils)
importlib.reload(src.model_utils)

from src.model_utils import predict_listing, get_map_context

prediction_test = predict_listing(
    latitude=36.72,
    longitude=-4.42,
    room_type="Entire home/apt",
    property_type_group="Apartamento",
    accommodates=4,
    bedrooms=2,
    bathrooms_num=1,
    minimum_nights=2,
    maximum_nights=365,
    instant_bookable=1,
    host_is_superhost=0,
    selected_amenities=[
        "has_wifi",
        "has_kitchen",
        "has_air_conditioning",
        "has_heating",
        "has_tv",
        "has_washer",
        "has_elevator",
    ],
)

print(
    f"Tarifa estimada: "
    f"{prediction_test['price']:.2f} € / noche"
)

print(
    "Shape de entrada:",
    prediction_test["features"].shape,
)

if prediction_test["features"].shape[1] != 164:
    raise ValueError(
        "La observación no contiene las 164 variables esperadas."
    )

map_test = get_map_context(
    latitude=36.72,
    longitude=-4.42,
    radius_km=1.0,
)

print("Distrito:", map_test["district"])
print("Competidores H3:", len(map_test["competitors"]))
print("Alojamientos a ≤1 km:", len(map_test["nearby"]))

### 5.1. Comprobación SHAP

Se conserva una única explicación local para verificar que el componente interpretativo funciona correctamente.

Como el modelo final predice `price` directamente, las contribuciones SHAP se encuentran en la misma escala de salida del modelo y pueden interpretarse aproximadamente en euros.


In [ ]:
import shap

shap.plots.waterfall(
    prediction_test["shap_explanation"],
    max_display=15,
)

## 6. RAG de producción

El RAG utilizado por la aplicación se encuentra en `src/rag_utils.py`
y se integra con la interfaz a través de FastAPI.

Configuración final:

- retrieval v2.2;
- prompt v3;
- `mistral:7b`;
- 4 fragmentos de contexto;
- ChromaDB `chroma_vut_malaga_v2`.

Este notebook de Streamlit no vuelve a escribir `src/rag_utils.py`, para evitar
sobrescribir accidentalmente la versión validada.

In [ ]:
# Comprobación de que el módulo RAG final existe.
RAG_UTILS_PATH = SRC_DIR / "rag_utils.py"

if not RAG_UTILS_PATH.exists():
    raise FileNotFoundError(
        "No existe src/rag_utils.py. "
        "Ejecuta antes el notebook final de FastAPI (Fase 5.1)."
    )

rag_source = RAG_UTILS_PATH.read_text(
    encoding="utf-8"
)

required_markers = [
    'FINAL_LLM = "mistral:7b"',
    "def retrieve_documents(",
    "def ask_rag(",
    "FINAL_TOP_K = 4",
]

missing_markers = [
    marker
    for marker in required_markers
    if marker not in rag_source
]

if missing_markers:
    raise RuntimeError(
        "src/rag_utils.py no parece corresponder "
        "al RAG final v2.2. Faltan: "
        + ", ".join(missing_markers)
    )

print(
    "OK: src/rag_utils.py corresponde "
    "al RAG final v2.2 + Mistral 7B."
)

## 7. Validación del backend

No se carga Chroma ni Mistral dentro de Streamlit. La comprobación se hace
contra FastAPI para mantener el desacoplamiento.

In [ ]:
from src.api_client import health_api

backend_status = health_api()

display(
    backend_status
)

print(
    "OK: Streamlit puede comunicarse con FastAPI."
)

## 8. Generación de la aplicación final

`app.py` funciona como capa de presentación y consume FastAPI mediante
`src/api_client.py`.

```text
Streamlit
    ↓ HTTP / JSON
FastAPI
├── /predict → XGBoost + SHAP
└── /rag     → RAG v2.2 + Mistral 7B
```

El timeout largo del RAG se gestiona en `src/api_client.py`, generado por la
Fase 5.1. Streamlit no importa ni ejecuta directamente el LLM.

In [ ]:
APP_CODE = '\nfrom pathlib import Path\n\nimport streamlit as st\nimport streamlit.components.v1 as components\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport shap\nimport folium\nimport h3\n\nfrom src.model_utils import get_map_context\nfrom src.api_client import (\n    health_api,\n    predict_api,\n    rag_api,\n)\n\n\n# ============================================================\n# CONFIGURACIÓN GENERAL\n# ============================================================\n\nst.set_page_config(\n    page_title="VUT Málaga · Sistema inteligente",\n        layout="wide",\n    initial_sidebar_state="expanded",\n)\n\n\n# ============================================================\n# ESTILOS\n# ============================================================\n\nst.markdown(\n    """\n    <style>\n        .stApp {\n            background:\n                radial-gradient(circle at top left, rgba(37,99,235,.08), transparent 28%),\n                radial-gradient(circle at top right, rgba(14,165,233,.07), transparent 24%),\n                #f7f9fc;\n        }\n\n        .block-container {\n            max-width: 1450px;\n            padding-top: 1.6rem;\n            padding-bottom: 3rem;\n        }\n\n        [data-testid="stSidebar"] {\n            background: linear-gradient(180deg, #0f172a 0%, #172554 100%);\n        }\n\n        [data-testid="stSidebar"] * {\n            color: #f8fafc;\n        }\n\n        .hero {\n            padding: 2.1rem 2.3rem;\n            border-radius: 24px;\n            background: linear-gradient(120deg, #0f172a 0%, #1e3a8a 55%, #0369a1 100%);\n            box-shadow: 0 16px 38px rgba(15,23,42,.18);\n            margin-bottom: 1.3rem;\n        }\n\n        .hero-eyebrow {\n            color: #bae6fd;\n            font-size: .82rem;\n            font-weight: 700;\n            letter-spacing: .11em;\n            text-transform: uppercase;\n            margin-bottom: .45rem;\n        }\n\n        .hero-title {\n            color: white;\n            font-size: 2.15rem;\n            line-height: 1.08;\n            font-weight: 800;\n            margin-bottom: .65rem;\n        }\n\n        .hero-copy {\n            color: #dbeafe;\n            font-size: 1.02rem;\n            max-width: 850px;\n            margin: 0;\n        }\n\n        .section-kicker {\n            color: #2563eb;\n            font-size: .78rem;\n            font-weight: 800;\n            text-transform: uppercase;\n            letter-spacing: .09em;\n            margin-bottom: .2rem;\n        }\n\n        .result-card {\n            border-radius: 22px;\n            padding: 1.7rem 1.9rem;\n            background: linear-gradient(135deg, #eff6ff 0%, #ffffff 72%);\n            border: 1px solid #bfdbfe;\n            box-shadow: 0 12px 30px rgba(37,99,235,.10);\n            margin: .4rem 0 1.2rem 0;\n        }\n\n        .result-label {\n            color: #475569;\n            font-weight: 650;\n            font-size: .9rem;\n            text-transform: uppercase;\n            letter-spacing: .06em;\n        }\n\n        .result-price {\n            color: #0f172a;\n            font-size: 3rem;\n            line-height: 1;\n            font-weight: 850;\n            margin: .35rem 0;\n        }\n\n        .result-sub {\n            color: #64748b;\n            font-size: .92rem;\n        }\n\n        div[data-testid="stMetric"] {\n            background: rgba(255,255,255,.84);\n            border: 1px solid #e2e8f0;\n            padding: .95rem 1rem;\n            border-radius: 16px;\n            box-shadow: 0 5px 16px rgba(15,23,42,.04);\n        }\n\n        div[data-testid="stForm"] {\n            border: 1px solid #dbe4f0;\n            background: rgba(255,255,255,.78);\n            border-radius: 20px;\n            padding: 1.1rem 1.1rem .6rem 1.1rem;\n            box-shadow: 0 8px 24px rgba(15,23,42,.04);\n        }\n\n        .rag-answer {\n            background: white;\n            border: 1px solid #dbe4f0;\n            border-left: 5px solid #2563eb;\n            border-radius: 16px;\n            padding: 1.2rem 1.3rem;\n            box-shadow: 0 6px 18px rgba(15,23,42,.05);\n            margin-top: .6rem;\n        }\n\n        .mini-note {\n            color: #64748b;\n            font-size: .86rem;\n        }\n\n        .market-card {\n            background: rgba(255,255,255,.88);\n            border: 1px solid #e2e8f0;\n            border-radius: 16px;\n            padding: 1rem 1.1rem;\n            box-shadow: 0 5px 16px rgba(15,23,42,.04);\n            height: 100%;\n        }\n\n        .market-label {\n            color: #64748b;\n            font-size: .78rem;\n            font-weight: 750;\n            letter-spacing: .05em;\n            text-transform: uppercase;\n            margin-bottom: .3rem;\n        }\n\n        .market-value {\n            color: #0f172a;\n            font-size: 1.55rem;\n            font-weight: 800;\n            line-height: 1.15;\n        }\n\n        .market-sub {\n            color: #64748b;\n            font-size: .82rem;\n            margin-top: .3rem;\n        }\n\n        .stTabs [data-baseweb="tab-list"] {\n            gap: .45rem;\n            background: rgba(255,255,255,.72);\n            padding: .35rem;\n            border-radius: 14px;\n            border: 1px solid #e2e8f0;\n        }\n\n        .stTabs [data-baseweb="tab"] {\n            border-radius: 10px;\n            padding-left: 1rem;\n            padding-right: 1rem;\n        }\n    </style>\n    """,\n    unsafe_allow_html=True,\n)\n\n\n\n# ============================================================\n# HELPERS API\n# ============================================================\n\ndef shap_from_api(shap_payload):\n    return shap.Explanation(\n        values=np.asarray(\n            shap_payload["values"],\n            dtype=float,\n        ),\n        base_values=float(\n            shap_payload["base_value"]\n        ),\n        data=np.asarray(\n            shap_payload["data"],\n            dtype=object,\n        ),\n        feature_names=list(\n            shap_payload["feature_names"]\n        ),\n    )\n\n\ndef api_status():\n    try:\n        result = health_api(timeout=3)\n        return result.get("status") == "ok"\n    except Exception:\n        return False\n\n\n# ============================================================\n# SIDEBAR\n# ============================================================\n\nwith st.sidebar:\n    st.markdown("## VUT Málaga")\n    st.caption("Sistema inteligente de apoyo a la estimación y consulta normativa")\n\n    st.divider()\n\n    backend_ok = api_status()\n\n    if backend_ok:\n        st.success("API REST conectada")\n    else:\n        st.error("API REST no disponible")\n\n    st.markdown("### Motor predictivo")\n    st.markdown("**Modelo:** XGBoost")\n    st.markdown("**Objetivo:** tarifa por noche")\n    st.markdown("**Validación:** holdout espacial 80/20")\n    st.markdown("**MAE test:** 33,20 €")\n\n    st.divider()\n\n    st.markdown("### Motor documental")\n    st.markdown("**Retrieval:** ChromaDB + embeddings")\n    st.markdown("**LLM local:** Ollama")\n    st.markdown("**Ámbito:** VUT Málaga / Andalucía")\n\n    st.divider()\n    st.caption(\n        "TFM · Sistema predictivo, geoespacial y conversacional "\n        "para viviendas de uso turístico."\n    )\n\n\n# ============================================================\n# CABECERA\n# ============================================================\n\nst.markdown(\n    """\n    <div class="hero">\n        <div class="hero-eyebrow">Trabajo Fin de Máster · Málaga</div>\n        <div class="hero-title">Sistema inteligente para viviendas de uso turístico</div>\n        <p class="hero-copy">\n            Estimación de tarifa mediante Machine Learning, análisis geoespacial\n            e interpretación SHAP, junto con un consultor documental basado en RAG.\n        </p>\n    </div>\n    """,\n    unsafe_allow_html=True,\n)\n\ntab1, tab2 = st.tabs([\n    "Estimador de tarifa",\n    "Consultor normativo",\n])\n\nif not api_status():\n    st.warning(\n        "La interfaz está disponible, pero la API REST no responde. "\n        "Arranca Uvicorn en el puerto 8000 para habilitar predicciones y consultas RAG."\n    )\n\n\n# ============================================================\n# PANEL 1\n# ============================================================\n\nwith tab1:\n\n    left_intro, right_intro = st.columns([2.2, 1])\n\n    with left_intro:\n        st.markdown(\n            \'<div class="section-kicker">Motor predictivo</div>\',\n            unsafe_allow_html=True,\n        )\n        st.subheader("Configura el alojamiento")\n        st.write(\n            "Introduce los datos básicos de la vivienda. El sistema completa "\n            "automáticamente las variables históricas y geoespaciales necesarias "\n            "para generar la estimación."\n        )\n\n    with right_intro:\n        st.info(\n            "La tarifa es una estimación orientativa basada en patrones históricos. "\n            "No representa una recomendación comercial vinculante."\n        )\n\n    with st.form("prediction_form"):\n\n        st.markdown("#### 📍 Ubicación")\n        col1, col2 = st.columns(2)\n\n        with col1:\n            latitude = st.number_input(\n                "Latitud",\n                value=36.7200,\n                format="%.6f",\n                help="Coordenada de la vivienda dentro del municipio de Málaga.",\n            )\n\n        with col2:\n            longitude = st.number_input(\n                "Longitud",\n                value=-4.4200,\n                format="%.6f",\n            )\n\n        st.markdown("#### 🏡 Características principales")\n        col1, col2, col3 = st.columns(3)\n\n        with col1:\n            room_type = st.selectbox(\n                "Tipo de alojamiento",\n                [\n                    "Entire home/apt",\n                    "Private room",\n                    "Shared room",\n                    "Hotel room",\n                ],\n            )\n\n            property_type_group = st.selectbox(\n                "Tipo de propiedad",\n                [\n                    "Apartamento",\n                    "Casa_Villa",\n                    "Habitacion",\n                    "Otros",\n                ],\n            )\n\n        with col2:\n            accommodates = st.number_input(\n                "Capacidad máxima",\n                min_value=1,\n                max_value=16,\n                value=4,\n                step=1,\n            )\n\n            bedrooms = st.number_input(\n                "Dormitorios",\n                min_value=0,\n                max_value=15,\n                value=2,\n                step=1,\n            )\n\n        with col3:\n            bathrooms_num = st.number_input(\n                "Número de baños",\n                min_value=0.0,\n                max_value=10.0,\n                value=1.0,\n                step=0.5,\n            )\n\n            host_is_superhost = st.checkbox(\n                "⭐ El anfitrión es Superhost",\n                value=False,\n            )\n\n        st.markdown("#### 📅 Condiciones de reserva")\n        col1, col2, col3 = st.columns(3)\n\n        with col1:\n            minimum_nights = st.number_input(\n                "Noches mínimas",\n                min_value=1,\n                max_value=30,\n                value=2,\n                step=1,\n            )\n\n        with col2:\n            maximum_nights = st.number_input(\n                "Noches máximas",\n                min_value=1,\n                max_value=1125,\n                value=365,\n                step=1,\n            )\n\n        with col3:\n            instant_bookable = st.checkbox(\n                "⚡ Reserva instantánea",\n                value=True,\n            )\n\n        st.markdown("#### ✨ Equipamiento")\n\n        amenity_labels = {\n            "has_wifi": "📶 Wifi",\n            "has_kitchen": "🍳 Cocina",\n            "has_air_conditioning": "❄️ Aire acondicionado",\n            "has_heating": "🔥 Calefacción",\n            "has_parking": "🚗 Parking",\n            "has_pool": "🏊 Piscina",\n            "has_washer": "🧺 Lavadora",\n            "has_dryer": "♨️ Secadora",\n            "has_tv": "📺 Televisión",\n            "has_balcony_or_terrace": "🌿 Balcón o terraza",\n            "has_sea_view": "🌊 Vistas al mar",\n            "has_workspace": "💻 Zona de trabajo",\n            "has_elevator": "🛗 Ascensor",\n            "has_pets_allowed": "🐾 Mascotas",\n            "has_crib": "👶 Cuna",\n            "has_bbq": "🔥 Barbacoa",\n            "has_gym": "🏋️ Gimnasio",\n            "has_hot_tub": "🫧 Jacuzzi",\n            "has_breakfast": "☕ Desayuno",\n        }\n\n        selected_amenities = []\n        amenity_columns = st.columns(4)\n\n        for i, (amenity, label) in enumerate(\n            amenity_labels.items()\n        ):\n            with amenity_columns[i % 4]:\n                checked = st.checkbox(\n                    label,\n                    key=f"amenity_{amenity}",\n                )\n                if checked:\n                    selected_amenities.append(amenity)\n\n        submitted = st.form_submit_button(\n            "Calcular tarifa estimada",\n            use_container_width=True,\n            type="primary",\n        )\n\n    # Calcular solo cuando se envía el formulario.\n    # Cambiar el radio del mapa no vuelve a ejecutar XGBoost ni SHAP.\n    if submitted:\n        try:\n            with st.spinner(\n                "Enviando datos a la API y estimando tarifa..."\n            ):\n                payload = {\n                    "latitude": latitude,\n                    "longitude": longitude,\n                    "room_type": room_type,\n                    "property_type_group": property_type_group,\n                    "accommodates": accommodates,\n                    "bedrooms": bedrooms,\n                    "bathrooms_num": bathrooms_num,\n                    "minimum_nights": minimum_nights,\n                    "maximum_nights": maximum_nights,\n                    "instant_bookable": int(instant_bookable),\n                    "host_is_superhost": int(host_is_superhost),\n                    "selected_amenities": selected_amenities,\n                }\n\n                api_result = predict_api(\n                    payload,\n                    timeout=120,\n                )\n\n                result = {\n                    "price": float(\n                        api_result["price"]\n                    ),\n                    "features": api_result["features"],\n                    "shap_explanation": shap_from_api(\n                        api_result["shap"]\n                    ),\n                }\n\n            st.session_state["prediction_result"] = result\n            st.session_state["prediction_coordinates"] = {\n                "latitude": latitude,\n                "longitude": longitude,\n            }\n\n        except Exception as error:\n            st.error(\n                f"No se ha podido realizar la predicción: {error}"\n            )\n\n    if "prediction_result" in st.session_state:\n\n        result = st.session_state["prediction_result"]\n        coordinates = st.session_state["prediction_coordinates"]\n\n        result_latitude = coordinates["latitude"]\n        result_longitude = coordinates["longitude"]\n        features = result["features"]\n\n        st.divider()\n\n        st.markdown(\n            \'<div class="section-kicker">Resultado del modelo</div>\',\n            unsafe_allow_html=True,\n        )\n\n        result_col, context_col = st.columns([1.15, 1.85])\n\n        with result_col:\n            st.markdown(\n                f"""\n                <div class="result-card">\n                    <div class="result-label">Tarifa estimada</div>\n                    <div class="result-price">{result[\'price\']:.2f} €</div>\n                    <div class="result-sub">por noche · estimación XGBoost</div>\n                </div>\n                """,\n                unsafe_allow_html=True,\n            )\n\n        with context_col:\n            c1, c2, c3 = st.columns(3)\n\n            with c1:\n                st.metric(\n                    "Distrito",\n                    str(features["district"]),\n                )\n\n            with c2:\n                st.metric(\n                    "Playa más cercana",\n                    f"{features[\'distance_nearest_beach_km\']:.2f} km",\n                )\n\n            with c3:\n                st.metric(\n                    "Centro histórico",\n                    f"{features[\'distance_nearest_historic_center_km\']:.2f} km",\n                )\n\n            c1, c2 = st.columns(2)\n\n            with c1:\n                st.metric(\n                    "Competidores H3",\n                    f"{int(features[\'h3_res8_competitor_count\']):,}",\n                )\n\n            with c2:\n                st.metric(\n                    "Densidad H3",\n                    f"{features[\'h3_res8_density_km2\']:.1f} aloj./km²",\n                )\n\n        # ========================================================\n        # MAPA\n        # ========================================================\n\n        st.markdown(\n            \'<div class="section-kicker">Análisis geoespacial</div>\',\n            unsafe_allow_html=True,\n        )\n        st.subheader("Mapa de competencia y proximidad")\n\n        map_control, map_info = st.columns([1.2, 2.8])\n\n        with map_control:\n            radius_m = st.select_slider(\n                "Radio de análisis",\n                options=[500, 1000, 2000],\n                value=1000,\n                format_func=lambda x: (\n                    f"{x} m"\n                    if x < 1000\n                    else f"{x / 1000:.0f} km"\n                ),\n            )\n\n        radius_km = radius_m / 1000\n\n        map_context = get_map_context(\n            latitude=result_latitude,\n            longitude=result_longitude,\n            radius_km=radius_km,\n        )\n\n        competitors = map_context["competitors"]\n        nearby = map_context["nearby"]\n\n        with map_info:\n            m1, m2 = st.columns(2)\n            with m1:\n                st.metric(\n                    "En la misma celda H3",\n                    f"{len(competitors):,}",\n                )\n            with m2:\n                st.metric(\n                    f"A ≤ {radius_m} m",\n                    f"{len(nearby):,}",\n                )\n\n        # ========================================================\n        # COMPARACIÓN CON EL MERCADO CERCANO\n        # ========================================================\n\n        st.markdown("#### Comparación con el mercado cercano")\n\n        nearby_prices = (\n            nearby["price"]\n            .dropna()\n            .astype(float)\n        )\n\n        if len(nearby_prices) > 0:\n            nearby_mean = float(\n                nearby_prices.mean()\n            )\n\n            nearby_median = float(\n                nearby_prices.median()\n            )\n\n            price_percentile = float(\n                (\n                    nearby_prices\n                    <= float(result["price"])\n                ).mean()\n                * 100\n            )\n\n            median_difference_pct = (\n                (\n                    float(result["price"])\n                    - nearby_median\n                )\n                / nearby_median\n                * 100\n                if nearby_median > 0\n                else np.nan\n            )\n\n            if np.isnan(median_difference_pct):\n                comparison_text = "Sin referencia comparable"\n            elif median_difference_pct >= 0:\n                comparison_text = (\n                    f"{median_difference_pct:.1f}% "\n                    "sobre la mediana"\n                )\n            else:\n                comparison_text = (\n                    f"{abs(median_difference_pct):.1f}% "\n                    "por debajo de la mediana"\n                )\n\n            mc1, mc2, mc3, mc4 = st.columns(4)\n\n            with mc1:\n                st.markdown(\n                    f\'\'\'\n                    <div class="market-card">\n                        <div class="market-label">Tarifa estimada</div>\n                        <div class="market-value">{result["price"]:.2f} €</div>\n                        <div class="market-sub">Modelo XGBoost</div>\n                    </div>\n                    \'\'\',\n                    unsafe_allow_html=True,\n                )\n\n            with mc2:\n                st.markdown(\n                    f\'\'\'\n                    <div class="market-card">\n                        <div class="market-label">Mediana del entorno</div>\n                        <div class="market-value">{nearby_median:.2f} €</div>\n                        <div class="market-sub">{comparison_text}</div>\n                    </div>\n                    \'\'\',\n                    unsafe_allow_html=True,\n                )\n\n            with mc3:\n                st.markdown(\n                    f\'\'\'\n                    <div class="market-card">\n                        <div class="market-label">Media del entorno</div>\n                        <div class="market-value">{nearby_mean:.2f} €</div>\n                        <div class="market-sub">{len(nearby_prices):,} alojamientos comparables</div>\n                    </div>\n                    \'\'\',\n                    unsafe_allow_html=True,\n                )\n\n            with mc4:\n                st.markdown(\n                    f\'\'\'\n                    <div class="market-card">\n                        <div class="market-label">Percentil de precio</div>\n                        <div class="market-value">P{price_percentile:.0f}</div>\n                        <div class="market-sub">\n                            Aproximadamente {price_percentile:.0f}% del entorno\n                            tiene una tarifa igual o inferior\n                        </div>\n                    </div>\n                    \'\'\',\n                    unsafe_allow_html=True,\n                )\n\n        else:\n            st.info(\n                "No hay suficientes alojamientos cercanos para "\n                "calcular una comparación de mercado."\n            )\n\n        m = folium.Map(\n            location=[result_latitude, result_longitude],\n            zoom_start=14,\n            tiles="CartoDB positron",\n            control_scale=True,\n        )\n\n        district_gdf = map_context["district_gdf"]\n\n        if not district_gdf.empty:\n            district_layer = folium.FeatureGroup(\n                name="Distrito",\n                show=True,\n            )\n\n            folium.GeoJson(\n                district_gdf.__geo_interface__,\n                tooltip=folium.GeoJsonTooltip(\n                    fields=["district"],\n                    aliases=["Distrito:"],\n                ),\n            ).add_to(district_layer)\n\n            district_layer.add_to(m)\n\n        h3_group = folium.FeatureGroup(\n            name="Celda H3 res. 8",\n            show=True,\n        )\n\n        h3_cell = map_context["h3_res8"]\n        h3_boundary = h3.cell_to_boundary(h3_cell)\n\n        h3_polygon = [\n            [lat, lon]\n            for lat, lon in h3_boundary\n        ]\n\n        folium.Polygon(\n            locations=h3_polygon,\n            tooltip="Celda H3 res. 8",\n            fill=True,\n            fill_opacity=0.12,\n            weight=3,\n        ).add_to(h3_group)\n\n        h3_group.add_to(m)\n\n        competitors_group = folium.FeatureGroup(\n            name="Competidores H3",\n            show=True,\n        )\n\n        for _, competitor in competitors.iterrows():\n\n            popup_text = (\n                f"<b>Tarifa:</b> {competitor[\'price\']:.2f} €<br>"\n                f"<b>Tipo:</b> {competitor[\'room_type\']}<br>"\n                f"<b>Propiedad:</b> {competitor[\'property_type_group\']}"\n            )\n\n            folium.CircleMarker(\n                location=[\n                    competitor["latitude"],\n                    competitor["longitude"],\n                ],\n                radius=3,\n                tooltip="Alojamiento competidor",\n                popup=popup_text,\n                fill=True,\n                fill_opacity=0.60,\n                weight=1,\n            ).add_to(competitors_group)\n\n        competitors_group.add_to(m)\n\n        nearby_group = folium.FeatureGroup(\n            name=f"Alojamientos a ≤ {radius_m} m",\n            show=True,\n        )\n\n        folium.Circle(\n            location=[\n                result_latitude,\n                result_longitude,\n            ],\n            radius=radius_m,\n            tooltip=f"Radio de proximidad: {radius_m} m",\n            fill=False,\n            weight=2,\n        ).add_to(nearby_group)\n\n        for _, listing in nearby.iterrows():\n\n            popup_nearby = (\n                f"<b>Tarifa:</b> {listing[\'price\']:.2f} €<br>"\n                f"<b>Distancia:</b> {listing[\'distance_km\']:.2f} km<br>"\n                f"<b>Tipo:</b> {listing[\'room_type\']}<br>"\n                f"<b>Propiedad:</b> {listing[\'property_type_group\']}"\n            )\n\n            folium.CircleMarker(\n                location=[\n                    listing["latitude"],\n                    listing["longitude"],\n                ],\n                radius=2,\n                tooltip=(\n                    f"{listing[\'distance_km\']:.2f} km · "\n                    f"{listing[\'price\']:.0f} €"\n                ),\n                popup=popup_nearby,\n                fill=True,\n                fill_opacity=0.32,\n                weight=1,\n            ).add_to(nearby_group)\n\n        nearby_group.add_to(m)\n\n        folium.Marker(\n            location=[\n                result_latitude,\n                result_longitude,\n            ],\n            tooltip="Vivienda analizada",\n            popup=(\n                f"<b>Vivienda analizada</b><br>"\n                f"Tarifa estimada: {result[\'price\']:.2f} €"\n            ),\n            icon=folium.Icon(\n                icon="home",\n                prefix="fa",\n            ),\n        ).add_to(m)\n\n        folium.LayerControl(\n            position="topright",\n            collapsed=True,\n        ).add_to(m)\n\n        map_html = m.get_root().render()\n\n        components.html(\n            map_html,\n            height=610,\n            scrolling=False,\n        )\n\n        st.caption(\n            "Usa el selector de capas situado en la esquina superior derecha "\n            "para mostrar u ocultar distrito, celda H3 y alojamientos."\n        )\n\n        # ========================================================\n        # SHAP\n        # ========================================================\n\n        st.markdown(\n            \'<div class="section-kicker">Interpretabilidad</div>\',\n            unsafe_allow_html=True,\n        )\n\n        with st.expander(\n            "¿Por qué el modelo estima esta tarifa?",\n            expanded=True,\n        ):\n            st.write(\n                "La interpretación se divide en dos vistas: primero se muestran "\n                "los cinco factores con mayor impacto absoluto y después el "\n                "waterfall completo de la predicción."\n            )\n\n            explanation = result["shap_explanation"]\n\n            factor_names = np.asarray(\n                explanation.feature_names,\n                dtype=object,\n            )\n\n            factor_values = np.asarray(\n                explanation.values,\n                dtype=float,\n            ).reshape(-1)\n\n            # Comprobación defensiva por si SHAP devuelve una estructura inesperada.\n            if factor_names.shape[0] != factor_values.shape[0]:\n                st.warning(\n                    "No se ha podido construir el resumen de factores SHAP "\n                    "por una diferencia entre nombres y contribuciones."\n                )\n\n            else:\n                top_factor_indices = np.argsort(\n                    np.abs(factor_values)\n                )[-5:]\n\n                top_names = []\n                top_values = []\n\n                for factor_index in top_factor_indices:\n                    clean_name = str(\n                        factor_names[factor_index]\n                    )\n\n                    clean_name = (\n                        clean_name\n                        .replace("numeric__", "")\n                        .replace("categorical__", "")\n                        .replace("_", " ")\n                    )\n\n                    top_names.append(clean_name)\n                    top_values.append(\n                        float(\n                            factor_values[factor_index]\n                        )\n                    )\n\n                st.markdown("#### Factores con mayor impacto")\n\n                fig_top, ax_top = plt.subplots(\n                    figsize=(9, 4.6)\n                )\n\n                y_positions = np.arange(\n                    len(top_names)\n                )\n\n                ax_top.barh(\n                    y_positions,\n                    top_values,\n                )\n\n                ax_top.set_yticks(\n                    y_positions\n                )\n\n                ax_top.set_yticklabels(\n                    top_names\n                )\n\n                ax_top.axvline(\n                    0,\n                    linewidth=1,\n                )\n\n                ax_top.set_xlabel(\n                    "Impacto sobre la tarifa estimada (€)"\n                )\n\n                ax_top.set_ylabel("")\n\n                ax_top.grid(\n                    axis="x",\n                    alpha=0.18,\n                )\n\n                for position, value in zip(\n                    y_positions,\n                    top_values,\n                ):\n                    offset = 0.35 if value >= 0 else -0.35\n                    alignment = "left" if value >= 0 else "right"\n\n                    ax_top.text(\n                        value + offset,\n                        position,\n                        f"{value:+.2f} €",\n                        va="center",\n                        ha=alignment,\n                        fontsize=9,\n                    )\n\n                fig_top.tight_layout()\n\n                st.pyplot(\n                    fig_top,\n                    clear_figure=True,\n                    use_container_width=True,\n                )\n\n                st.caption(\n                    "Valores positivos elevan la tarifa estimada; "\n                    "valores negativos la reducen."\n                )\n\n            st.markdown("#### Explicación completa")\n\n            fig_waterfall = plt.figure(\n                figsize=(10, 7)\n            )\n\n            shap.plots.waterfall(\n                explanation,\n                max_display=15,\n                show=False,\n            )\n\n            plt.gcf().set_size_inches(\n                10,\n                7,\n            )\n\n            plt.subplots_adjust(\n                top=0.93,\n                left=0.30,\n                right=0.95,\n                bottom=0.08,\n            )\n\n            st.pyplot(\n                plt.gcf(),\n                clear_figure=True,\n                use_container_width=True,\n            )\n\n            st.caption(\n                "Las contribuciones SHAP están expresadas en la misma escala "\n                "de salida del modelo y pueden interpretarse aproximadamente "\n                "como euros añadidos o restados a la estimación."\n            )\n\n\n# ============================================================\n# CALLBACK FAQ\n# ============================================================\n\ndef load_faq_question(question):\n    """\n    Carga una pregunta frecuente en el cuadro de texto.\n\n    Los callbacks de Streamlit se ejecutan antes de volver a renderizar\n    los widgets, por lo que podemos actualizar rag_question sin provocar\n    StreamlitAPIException ni lanzar automáticamente la consulta RAG.\n    """\n    st.session_state["rag_question"] = question\n\n\n# ============================================================\n# PANEL 2\n# ============================================================\n\nwith tab2:\n\n    st.markdown(\n        \'<div class="section-kicker">Asistente documental</div>\',\n        unsafe_allow_html=True,\n    )\n    st.header("Consultor de normativa VUT")\n\n    st.write(\n        "Consulta la documentación incorporada al proyecto sobre "\n        "viviendas de uso turístico en Málaga y Andalucía."\n    )\n\n    st.warning(\n        "La respuesta se genera a partir del corpus documental del TFM. "\n        "Es una herramienta informativa y no sustituye asesoramiento jurídico."\n    )\n\n    rag_main, rag_guide = st.columns(\n        [1.85, 1.15],\n        gap="large",\n    )\n\n    # --------------------------------------------------------\n    # COLUMNA PRINCIPAL\n    # --------------------------------------------------------\n\n    with rag_main:\n\n        st.markdown("### Realiza una consulta")\n\n        rag_question = st.text_area(\n            "Pregunta",\n            placeholder=(\n                "Escribe aquí una consulta sobre normativa, "\n                "limitaciones urbanísticas, requisitos o funcionamiento "\n                "de las viviendas de uso turístico..."\n            ),\n            height=135,\n            key="rag_question",\n        )\n\n        rag_submit = st.button(\n            "Consultar documentación",\n            type="primary",\n            key="rag_submit",\n            use_container_width=True,\n        )\n\n        st.caption(\n            "También puedes seleccionar una pregunta frecuente del panel derecho."\n        )\n\n        if rag_submit:\n\n            if not rag_question.strip():\n                st.warning(\n                    "Escribe una pregunta antes de realizar la consulta."\n                )\n\n            else:\n                with st.spinner(\n                    "Consultando la API documental y generando respuesta..."\n                ):\n                    try:\n                        api_rag_result = rag_api(\n                            rag_question.strip(),\n                            n_results=4,\n                            timeout=900,\n                        )\n\n                        rag_result = {\n                            "answer": api_rag_result["answer"],\n                            "sources": api_rag_result["sources"],\n                        }\n\n                        st.session_state["rag_result"] = rag_result\n                        st.session_state["rag_last_question"] = (\n                            rag_question.strip()\n                        )\n\n                    except Exception as exc:\n                        st.error(\n                            f"Error al consultar la API RAG: {exc}"\n                        )\n\n        if "rag_result" in st.session_state:\n\n            rag_result = st.session_state["rag_result"]\n\n            st.divider()\n\n            st.markdown(\n                \'<div class="section-kicker">Respuesta generada</div>\',\n                unsafe_allow_html=True,\n            )\n\n            st.subheader(\n                st.session_state.get(\n                    "rag_last_question",\n                    "Respuesta",\n                )\n            )\n\n            with st.container(border=True):\n                st.markdown(\n                    rag_result["answer"]\n                )\n\n            sources = rag_result["sources"]\n\n            if sources:\n\n                unique_sources = []\n                seen_sources = set()\n\n                for source in sources:\n                    key = (\n                        source.get("documento"),\n                        source.get("pagina"),\n                    )\n\n                    if key not in seen_sources:\n                        seen_sources.add(key)\n                        unique_sources.append(source)\n\n                with st.expander(\n                    "Documentos consultados",\n                    expanded=True,\n                ):\n                    for index, source in enumerate(\n                        unique_sources,\n                        start=1,\n                    ):\n                        st.markdown(\n                            f"**{index}. {source.get(\'documento\')}**  \\n"\n                            f"Página {source.get(\'pagina\')}"\n                        )\n\n    # --------------------------------------------------------\n    # PANEL DERECHO DE CONSULTAS GUIADAS\n    # --------------------------------------------------------\n\n    with rag_guide:\n\n        st.markdown("### Preguntas frecuentes")\n        st.caption(\n            "Selecciona una consulta para cargarla automáticamente "\n            "en el cuadro de pregunta."\n        )\n\n        faq_groups = {\n            "Requisitos de la vivienda": [\n                "¿Qué requisitos debe cumplir una vivienda de uso turístico?",\n                "¿Qué condiciones debe cumplir la vivienda para poder destinarse a uso turístico?",\n                "¿Qué obligaciones tiene la persona titular de una vivienda de uso turístico?",\n                "¿Qué requisitos de equipamiento se exigen a una vivienda de uso turístico?",\n            ],\n            "Urbanismo y limitaciones": [\n                "¿Qué ocurre cuando un barrio supera el 8% de viviendas de uso turístico?",\n                "¿Existen zonas de Málaga donde se limite la implantación de nuevas viviendas de uso turístico?",\n                "¿Cómo afecta el uso residencial del edificio a la implantación de una vivienda de uso turístico?",\n                "¿Qué criterios utiliza el Ayuntamiento de Málaga para limitar las viviendas de uso turístico?",\n            ],\n            "Normativa aplicable": [\n                "¿Qué establece la Instrucción 1/2024 sobre las viviendas de uso turístico?",\n                "¿Qué regula el Decreto 28/2016 sobre viviendas de uso turístico?",\n                "¿Qué cambios introduce el Decreto 31/2024?",\n                "¿Qué acordó el Pleno del Ayuntamiento de Málaga el 29 de mayo de 2025 sobre las viviendas de uso turístico?",\n            ],\n            "Comunidad y compatibilidad de usos": [\n                "¿Puede una comunidad de propietarios limitar o prohibir una vivienda de uso turístico?",\n                "¿Puede existir una vivienda de uso turístico en un edificio de uso residencial?",\n                "¿Cómo se relaciona la normativa turística con las limitaciones urbanísticas municipales?",\n            ],\n            "Control y cumplimiento": [\n                "¿Qué puede ocurrir si una vivienda de uso turístico incumple la normativa?",\n                "¿En qué casos puede dejar de considerarse válida una vivienda de uso turístico?",\n                "¿Qué controles puede realizar la administración sobre las viviendas de uso turístico?",\n            ],\n        }\n\n        for group_name, questions in faq_groups.items():\n\n            with st.expander(\n                group_name,\n                expanded=(group_name == "Requisitos de la vivienda"),\n            ):\n                for i, question in enumerate(questions):\n                    st.button(\n                        question,\n                        key=f"faq_{group_name}_{i}",\n                        use_container_width=True,\n                        on_click=load_faq_question,\n                        args=(question,),\n                    )\n\n        st.markdown("---")\n        st.caption(\n            "Si ninguna de estas preguntas coincide con tu duda, "\n            "puedes escribir una consulta libre en el panel principal."\n        )\n'
APP_PATH = PROJECT_ROOT / "app.py"
APP_PATH.write_text(
    APP_CODE,
    encoding="utf-8",
)

print("✅ Creado:", APP_PATH)

## 9. Validación estática de los archivos de producción

Se comprueba la sintaxis de los principales módulos de producción antes de iniciar Streamlit.

In [ ]:
import ast

production_files = [
    SRC_DIR / "geo_utils.py",
    SRC_DIR / "model_utils.py",
    SRC_DIR / "rag_utils.py",
    PROJECT_ROOT / "app.py",
]

for file_path in production_files:
    source = file_path.read_text(
        encoding="utf-8"
    )

    ast.parse(source)

    print(
        f" Sintaxis correcta: "
        f"{file_path.relative_to(PROJECT_ROOT)}"
    )

## 10. Dependencias reproducibles

El archivo `requirements.txt` recoge las principales dependencias y versiones
del entorno utilizado para validar la aplicación final.

## 11. Manual de instalación y ejecución

El README documenta la arquitectura, la preparación del entorno, la dependencia externa de Ollama, el cambio de LLM y las principales limitaciones del prototipo.

## 12. Verificación final de reproducibilidad

Se comprueba conjuntamente que están presentes los archivos y recursos que necesita la aplicación.

In [ ]:
final_checks = {
    "app.py":
        PROJECT_ROOT / "app.py",

    "requirements.txt":
        PROJECT_ROOT / "requirements.txt",

    "README.md":
        PROJECT_ROOT / "README.md",

    "src/geo_utils.py":
        SRC_DIR / "geo_utils.py",

    "src/model_utils.py":
        SRC_DIR / "model_utils.py",

    "src/rag_utils.py":
        SRC_DIR / "rag_utils.py",

    "Modelo XGBoost":
        MODEL_PATH,

    "Features":
        FEATURES_PATH,

    "GeoJSON":
        GEOJSON_PATH,

    "Vectorstore":
        VECTORSTORE_DIR,
}

all_ok = True

for name, path in final_checks.items():

    ok = path.exists()
    all_ok = all_ok and ok

    print(
        ("✅" if ok else "❌"),
        name,
    )

if not all_ok:
    raise FileNotFoundError(
        "La comprobación final ha detectado recursos ausentes."
    )

print()
print(" Estructura de producción completa")

## 13. Ejecución de Streamlit

Desde la carpeta raíz del proyecto:

```bash
python -m streamlit run app.py --server.fileWatcherType=none
```

Para utilizar otro modelo disponible en Ollama, puede definirse previamente `OLLAMA_MODEL`.

La ejecución de Streamlit se realiza desde una terminal; no es necesario lanzar un segundo proceso desde el notebook.

# Conclusión de la Fase 5

La interfaz Streamlit queda desacoplada de los motores inteligentes mediante una API REST FastAPI.

La arquitectura final separa tres responsabilidades:

```text
Capa de presentación
Streamlit
      ↓
Capa de servicios
FastAPI REST
      ↓
Capa de inteligencia
XGBoost + SHAP + ChromaDB + Ollama
```

La aplicación conserva la experiencia visual desarrollada previamente, pero las
predicciones y consultas documentales se realizan mediante peticiones HTTP/JSON.
Esta separación permite reutilizar el backend desde otras interfaces sin depender
directamente de Streamlit.